# Testing csprng on CW

# Clock glitching

## Setup

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'
SS_VER = 'SS_VER_2_1'

In [2]:
%run "Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 13294033                  to 35890172                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 0                         to 29538471                 
scope.clock.adc_rate                     changed from 0.0                       to 29538471.0        

In [47]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../firmware/mcu/csprng-test
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
arm-none-eabi-gcc (15:13.2.rel1-2) 13.2.1 20231009
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWLITEARM 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
Compiling:
Compiling:
Compiling:
-en     csprng.c ...
.
-en     csprng_hash.c ...
-en     fips202.c ...
Compiling:
-en     keccakf1600.c ...
.
Compiling:
.
Compiling:
.
.
-en     randombytes.c ...
.
-en     .././simpleserial/simpleserial.c ...
Compiling:
Compiling:
Compiling:
-en     .././hal/hal.c ...
-en     .././hal//stm32f3/stm32f3_hal.c ...
.
-en     .././hal//stm32f3/stm32f3_hal_lowlevel.c ...
.
Compiling:
Assembling: .././hal//stm32f3/stm32f3_startup.S
arm-none-eabi-gcc -c -mcpu=cortex-m4 -I. -x assembler-with-cpp -mthumb -mfloat-abi=soft -fmessage-length=0 -ffunction-sections -DF_CPU=737280

.././simpleserial/simpleserial.c: In function 'ss_puts':
.././simpleserial/simpleserial.c:82:17: warning: implicit declaration of function 'putch' [-Wimplicit-function-declaration]
   82 |                 putch(*x);
      |                 ^~~~~
.././simpleserial/simpleserial.c: In function 'simpleserial_get':
In file included from csprng_hash.c:34:
csprng_hash.h: In function 'csprng_fp_mat_trigger':
.././simpleserial/simpleserial.c:168:31: warning: implicit declaration of function 'getch' [-Wimplicit-function-declaration]
  168 |                 data_buf[i] = getch(); //PTR, cmd, scmd, len
      |                               ^~~~~
csprng_hash.h:145:13: warning: implicit declaration of function 'trigger_high' [-Wimplicit-function-declaration]
  145 |             trigger_high();
      |             ^~~~~~~~~~~~
csprng_hash.h:147:13: warning: implicit declaration of function 'trigger_low' [-Wimplicit-function-declaration]
  147 |             trigger_low();
      |             ^~~~~~~~~

-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
.
LINKING:
-en     csprng-CWLITEARM.elf ...
Memory region         Used Size  Region Size  %age Used
             RAM:        5472 B        40 KB     13.36%
             ROM:       13976 B       256 KB      5.33%
-e Done!
.
.
.
Creating load file for Flash: csprng-CWLITEARM.hex
arm-none-eabi-objcopy -O ihex -R .eeprom -R .fuse -R .lock -R .signature csprng-CWLITEARM.elf csprng-CWLITEARM.hex
.
Creating load file for Flash: csprng-CWLITEARM.bin
Creating load file for EEPROM: csprng-CWLITEARM.eep
.
arm-none-eabi-objcopy -O binary -R .eeprom -R .fuse -R .lock -R .signature csprng-CWLITEARM.elf csprng-CWLITEARM.bin
arm-none-eabi-objcopy -j .eeprom --set-section-flags=.eeprom="alloc,load" \
--change-section-lma .eeprom=0 --no-change-warnings -O ihex csprng-CWLITEARM.elf csprng-CWLITEARM.eep || exit 0
Creating Extended Listing: csprng-CWLITEARM.lss
Creating Symbol Table: csprng-CWLITEARM.sym
arm-none-eabi

In [48]:
fw_path = "../firmware/mcu/csprng-test/csprng-{}.hex".format(PLATFORM)
#prog = cw.programmers.STM32FProgrammer
cw.program_target(scope, prog, fw_path)
if SS_VER=='SS_VER_2_1':
    target.reset_comms()

Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 13975 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 13975 bytes


In [49]:
if PLATFORM == "CWLITEXMEGA":
    def reboot_flush():            
        scope.io.pdic = False
        time.sleep(0.1)
        scope.io.pdic = "high_z"
        time.sleep(0.1)
        #Flush garbage too
        target.flush()
else:
    def reboot_flush():            
        scope.io.nrst = False
        time.sleep(0.05)
        scope.io.nrst = "high_z"
        time.sleep(0.05)
        #Flush garbage too
        target.flush()

## Communication tests

In [50]:
msg = bytearray([0]*2) #simpleserial uses bytearrays
target.simpleserial_write('m', msg)

val = target.simpleserial_read_witherrors('r', 2, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val['full_response'])
print(val)

CWbytearray(b'00 72 02 00 00 60 00')
{'valid': True, 'payload': CWbytearray(b'00 00'), 'full_response': CWbytearray(b'00 72 02 00 00 60 00'), 'rv': bytearray(b'\x00')}


## Matrix export

In [51]:
n = 127
k = 76
rounds = (n-k)*k//38
def read_matrix():
    msg = bytearray([0]*38)
    data = bytearray()
    for i in range(rounds) :
        target.simpleserial_write('g', msg)
        buff = target.simpleserial_read('r', 38)
        data.extend(buff)
    return data

def matrix_of(data):
    res = [[0x00 for _ in range(n-k)] for _ in range(k)]
    for i in range(k) :
        for j in range(n-k) :
            if i*(n-k)+j :
                res[i][j]= data[i*(n-k)+j]
    return res

def pp_matrix(matrix) :
    s = [[str(e) for e in row] for row in matrix]
    lens = [max(map(len, col)) for col in zip(*s)]
    fmt = '\t'.join('{{:{}}}'.format(x) for x in lens)
    table = [fmt.format(*row) for row in s]
    print('\n'.join(table))
    
def export_matrix(matrix_bytes,offset, width, nb_tries,folder="./") : 
    filename = folder+"/mat{}_{}_{}".format(offset, width, nb_tries)
    with open(filename, 'wb') as fw:
        fw.write(matrix_bytes)

In [52]:
## test 
matrix_bytes = read_matrix()
pp_matrix(matrix_of(matrix_bytes))
export_matrix(matrix_bytes,0,0,0,"./exports")

0  	3  	95 	79 	126	11 	81 	4  	100	40 	78 	1  	66 	67 	99 	89 	76 	44 	25 	3  	117	110	123	121	70 	111	88 	18 	95 	98 	37 	53 	85 	45 	86 	114	54 	71 	121	96 	83 	97 	28 	4  	29 	108	54 	7  	24 	102	31 
105	115	21 	41 	119	113	60 	0  	122	38 	2  	113	109	124	117	20 	71 	76 	46 	68 	35 	95 	104	96 	10 	82 	41 	102	113	101	20 	43 	86 	36 	19 	98 	96 	89 	76 	126	39 	102	17 	81 	72 	102	72 	101	49 	92 	56 
74 	28 	2  	31 	110	20 	36 	16 	82 	61 	112	92 	84 	10 	76 	86 	122	40 	71 	78 	78 	121	99 	43 	7  	121	29 	41 	23 	72 	74 	29 	40 	98 	36 	104	104	66 	126	48 	29 	41 	30 	52 	106	94 	54 	16 	117	1  	24 
41 	104	6  	117	17 	93 	18 	33 	4  	30 	92 	53 	66 	48 	107	94 	41 	26 	6  	18 	0  	64 	75 	28 	57 	119	96 	13 	110	51 	113	58 	21 	113	84 	81 	59 	69 	15 	53 	28 	7  	65 	56 	55 	125	67 	3  	4  	64 	96 
12 	73 	48 	23 	69 	49 	76 	87 	68 	100	24 	123	57 	91 	31 	80 	14 	69 	111	49 	73 	54 	45 	34 	35 	64 	40 	0  	24 	107	46 	124	103	105	86 	68 	121	100	48 	101	40 	35 	43 	15 	22 	11 	

## Glitch configuration

In [53]:
gc = cw.GlitchController(groups=["success", "reset", "normal", "non_unique"], parameters=["width", "offset", "tries"])

In [54]:
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

IntText(value=0, description='non_unique count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='tries setting:', disabled=True, max=10.0, readout…

In [55]:
gc.glitch_plot(plotdots={ "reset":"xr", "non_unique":"*b", "success":"+g", "normal":None})

invalid response 
Timeout - no trigger 
Parameter name clashes for keys ['data'] 
invalid response 
invalid response 
invalid response

:DynamicMap   []
   :Overlay
      .Points.I   :Points   [width,offset]
      .Points.II  :Points   [width,offset]
      .Points.III :Points   [width,offset]

In [56]:
#Basic setup
# set glitch clock
scope.glitch.clk_src = "clkgen" 
scope.glitch.output = "clock_xor" # glitch_out = clk ^ glitch
scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called
scope.io.hs2 = "glitch"  # output glitch_out on the clock line
print(scope.glitch)

clk_src     = clkgen
mmcm_locked = True
width       = 5.859375
width_fine  = 0
offset      = 1.953125
offset_fine = 0
trigger_src = ext_single
arm_timing  = after_scope
ext_offset  = 8
repeat      = 5
output      = clock_xor



## Glitch loop

In [57]:
flipped_coords = []

In [60]:
import chipwhisperer.common.results.glitch as glitch
from tqdm.notebook import trange
import struct

scope.glitch.ext_offset = 8

gc.set_range("width", 2, 6)
gc.set_range("offset", -2, 2)
gc.set_global_step([8, 4, 2, 1])

scope.glitch.repeat = 5
gc.set_step("tries", 1)
scope.adc.timeout = 0.1

reboot_flush()

for glitch_setting in gc.glitch_values():
    
    # optional: you can speed up the loop by checking if the trigger never went low
    #           (the target never called trigger_low();) via scope.adc.state
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]

    scope.arm()
    
    target.simpleserial_write("m", bytearray([0,0]))
    
    ret = scope.capture()
    
    val = target.simpleserial_read_witherrors('r', 2, glitch_timeout=10)#For loop check
    
    # ###################
    # Add your code here
    # ###################
    
    if ret: #here the trigger never went high - sometimes the target is still crashed from a previous glitch
        print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
    else:
        # diff_num is the number of changed matrix values
        
        if val['valid'] == False :
            gc.add('reset')
            print("invalid response")
        else :        
            diff_num = int.from_bytes(val['payload'], byteorder='big', signed=False)
            single = int.from_bytes(val['rv'])
            if diff_num == 0: 
                #print('No change in the matrix')
                gc.add('normal')
            elif single != 0: #glitch!!!
                gc.add('success')
                print('Single fault matrix !!')
                flipped_coords.append(val['payload'])
                print(f'coords :',val['payload'])
                matrix_bytes = read_matrix()
                export_matrix(matrix_bytes,scope.glitch.offset,scope.glitch.width,0,"./exports")
            else :
                gc.add('non_unique')
                

(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b


invalid response


(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0a


Timeout - no trigger
invalid response


(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 08


...

Timeout - no trigger
invalid response


(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0a


Timeout - no trigger
invalid response


(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:731) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0a


Timeout - no trigger
invalid response


## Disconnecting

Disconnect from the hardware so it doesn't stay "in use" by this notebook : 

In [ ]:
scope.dis()
target.dis()

In [59]:
print(flipped_coords)

[]
